# PSDAT Interactive Lab — Jupyter edition

Three ways to work, all driving the **same validated engine** (`units.py` / `psdat_dae.m`):

1. **The full lab, embedded** (next cell) — the complete browser GUI *inside the notebook*: the
   single-line-diagram editor with the three benchmark systems (drag, connect, edit, run power flow
   on the drawing), ⚙ parameter sheets on every generator, exact small-signal analysis, the three
   disturbance classes, sweeps, POD design and Bode.
2. **Native widget panel** — pure `ipywidgets` controls for the core workflows.
3. **Scripted helpers** — `nb_linearize()`, `nb_simulate()`, `nb_sweep()` … for reproducible
   classroom exercises and homework.

Run from the `python/` folder of PSDAT (needs `numpy` + `matplotlib`; part 2 also needs `ipywidgets`).

## 0 · Setup — find the toolbox


In [ ]:
# ---- run this cell FIRST: locate the PSDAT python folder ----
# The toolbox is a SET of files (units.py, cases.py, psdat_gui.py, ...) that
# must stay together. This cell finds them even if you copied the notebook
# somewhere else. If the auto-search fails, set PSDAT_PATH yourself:
PSDAT_PATH = r""     # e.g. r"C:\Users\you\Desktop\Project_PSDAT\PSDAT\python"

import os, sys

def _find_psdat():
    home = os.path.expanduser("~")
    here = os.getcwd()
    cands = [PSDAT_PATH, here, os.path.join(here, "python"),
             os.path.join(here, "PSDAT", "python"),
             os.path.join(home, "Desktop", "Project_PSDAT", "PSDAT", "python"),
             os.path.join(home, "OneDrive", "Desktop", "Project_PSDAT", "PSDAT", "python"),
             os.path.join(home, "Desktop", "PSDAT", "python"),
             os.path.join(home, "PSDAT", "python")]
    for c in cands:
        if c and os.path.isfile(os.path.join(c, "units.py")) \
             and os.path.isfile(os.path.join(c, "psdat_gui.py")):
            return os.path.abspath(c)
    raise FileNotFoundError(
        "PSDAT python folder not found. Start Jupyter from PSDAT/python, or set "
        "PSDAT_PATH at the top of this cell to that folder. (It must contain "
        "units.py, cases.py, psdat_gui.py, ... - keep the files together; copying "
        "psdat_gui.py alone breaks its imports.)")

_P = _find_psdat()
if _P not in sys.path:
    sys.path.insert(0, _P)
os.chdir(_P)          # scenario figures etc. save next to the toolbox
print("using PSDAT folder:", _P)


## 1 · The full lab, embedded

In [ ]:
# ---- launch the PSDAT engine server and embed the FULL lab ----
import threading, socket, time
from IPython.display import IFrame, display
import psdat_gui as G

def _free_port(start=8642):
    for p in range(start, start + 60):
        with socket.socket() as s:
            if s.connect_ex(('127.0.0.1', p)) != 0:
                return p
    return start

try:
    _PORT                                   # already launched in this kernel?
except NameError:
    _PORT = _free_port()
    _SRV = G.ThreadingHTTPServer(('127.0.0.1', _PORT), G.H)
    threading.Thread(target=_SRV.serve_forever, daemon=True).start()
    time.sleep(0.5)

print(f"PSDAT Interactive Lab is running -> http://localhost:{_PORT}")
print("(embedded below; on a remote JupyterHub open that URL through your port forwarding)")
IFrame(f"http://localhost:{_PORT}", width="100%", height=780)


## 2 · Engine helpers
Run this cell first — it defines the notebook state `NB` (system, unit mix, parameter overrides)
and the pure functions that both the widgets and your own scripts use.

In [ ]:
# ---- engine-facing helpers (pure functions; the widgets below wire onto these) ----
import numpy as np
import matplotlib.pyplot as plt
import psdat_gui as G

NB = {"system": "IEEE9", "mix": ["SG", "SG", "SG"], "prm": {}}   # notebook state

def nb_payload(extra=None):
    p = dict(extra or {})
    p["system"] = NB["system"]; p["mix"] = list(NB["mix"]); p["prm"] = NB["prm"]
    return p

def nb_set_system(name):
    NB["system"] = name
    NB["mix"] = ["SG"] * G.api_meta(None)["m"][name]
    NB["prm"] = {}

def nb_set_override(unit, param, value):
    NB["prm"].setdefault(str(unit), {})[param] = value

def nb_clear_overrides(unit=None):
    if unit is None: NB["prm"] = {}
    else: NB["prm"].pop(str(unit), None)

def nb_linearize(show=True):
    r = G.api_linearize(nb_payload())
    if "error" in r: print("ERROR:", r["error"]); return r
    if show:
        fig, ax = plt.subplots(figsize=(6, 4.2))
        ev = np.array(r["ev"])
        st = ev[:, 0] <= 1e-6
        ax.plot(ev[st, 0], ev[st, 1], "x", color="#1f3b73", ms=8, mew=1.6)
        if (~st).any(): ax.plot(ev[~st, 0], ev[~st, 1], "x", color="#b42318", ms=9, mew=2)
        ax.axvline(0, color="#b42318", lw=1); ax.grid(alpha=0.3)
        ax.set_xlim(max(-12, ev[:, 0].min() - 0.5), max(0.5, ev[:, 0].max() + 0.3))
        ax.set_xlabel("Real (1/s)"); ax.set_ylabel("Imag (rad/s)")
        ax.set_title(f"{NB['system']} [{','.join(NB['mix'])}] — "
                     f"H_sys={r['Heff']} s, IBR {r['pen']}%, {r['unstable']} unstable")
        plt.show()
        print(f"{'f (Hz)':>8} {'zeta (%)':>9}   eigenvalue")
        for m in r["modes"][:12]:
            flag = "  <-- UNSTABLE" if m["z"] < 0 else ("  (weak)" if m["z"] < 5 else "")
            print(f"{m['f']:8.3f} {m['z']:9.2f}   {m['re']:.3f} ± {m['im']:.3f}j{flag}")
    return r

def nb_mode_detail(r, k=0):
    m = r["modes"][k]
    print(f"mode {m['f']} Hz, {m['z']} % damping — participation:")
    for name, v in m["part"]:
        print(f"   {name:10s} {'█' * int(round(v * 24)):24s} {v:.2f}")
    if m.get("shape"):
        fig, ax = plt.subplots(figsize=(3.6, 3.6), subplot_kw=dict(aspect="equal"))
        th = np.linspace(0, 2 * np.pi, 100)
        ax.plot(np.cos(th), np.sin(th), color="#e3e7ef")
        for lbl, re_, im_ in m["shape"]:
            ax.annotate("", xy=(re_, im_), xytext=(0, 0),
                        arrowprops=dict(arrowstyle="->", lw=2))
            ax.text(re_ * 1.12, im_ * 1.12, lbl, ha="center", fontsize=10)
        ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3); ax.axis("off")
        ax.set_title("mode shape (speed phasors)")
        plt.show()

def nb_simulate(kind="load", loc=8, mag=0.15, t1=1.0, t2="", tsim=12.0,
                watch=(), show=True):
    r = G.api_simulate(nb_payload({"watch": list(watch),
        "dist": dict(kind=kind, loc=loc, loc2=loc, mag=mag, t1=t1, t2=t2, tsim=tsim)}))
    if "error" in r: print("ERROR:", r["error"]); return r
    if show:
        t = np.array(r["t"])
        nplots = 1 + (1 if r["speeds"] else 0) + (1 if r["watch"] else 0)
        fig, axs = plt.subplots(nplots, 1, figsize=(7, 2.9 * nplots), sharex=True)
        axs = np.atleast_1d(axs); i = 0
        if r["fCOI"]:
            axs[i].plot(t, r["fCOI"], color="#1f3b73"); axs[i].grid(alpha=0.3)
            axs[i].set_ylabel("f_COI (Hz)")
            mt = r["metrics"]
            axs[i].set_title(f"nadir {mt['nadir']} Hz · RoCoF {mt['rocof']} Hz/s · "
                             f"final {mt['fend']} Hz", fontsize=10)
            i += 1
        if r["speeds"]:
            for nm, ys in r["speeds"].items(): axs[i].plot(t, ys, label=nm)
            axs[i].legend(fontsize=8); axs[i].grid(alpha=0.3)
            axs[i].set_ylabel("unit f (Hz)"); i += 1
        if r["watch"]:
            for nm, ys in r["watch"].items(): axs[i].plot(t, ys, label=nm)
            axs[i].legend(fontsize=8); axs[i].grid(alpha=0.3)
            axs[i].set_ylabel("watched")
        axs[-1].set_xlabel("time (s)")
        plt.tight_layout(); plt.show()
    return r

def nb_sweep(unit=1, param="Hv", vfrom=2, vto=12, n=7, band=(0.2, 3.0), show=True):
    r = G.api_sweep(nb_payload({"unit": unit, "param": param, "vfrom": vfrom,
                                "vto": vto, "n": n, "band": list(band)}))
    if "error" in r: print("ERROR:", r["error"]); return r
    if show:
        fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.8))
        nv = len(r["loci"])
        for i, ev in enumerate(r["loci"]):
            f = 0.25 + 0.75 * i / max(nv - 1, 1)
            ev = np.array(ev)
            if len(ev): ax[0].scatter(ev[:, 0], ev[:, 1], s=8 + 14 * f,
                                       color=(0.12, 0.23, 0.45, f))
        ax[0].axvline(0, color="#b42318", lw=1); ax[0].grid(alpha=0.3)
        ax[0].set_xlabel("Real (1/s)"); ax[0].set_ylabel("Imag (rad/s)")
        ax[0].set_title(f"root locus: {param} (light -> dark)")
        cv = np.array([[c[0], c[1]] for c in r["curve"] if c[1] is not None], float)
        if len(cv): ax[1].plot(cv[:, 0], cv[:, 1], "o-", color="#1f3b73")
        ax[1].grid(alpha=0.3); ax[1].set_xlabel(param)
        ax[1].set_ylabel("least damping in band (%)")
        plt.tight_layout(); plt.show()
    return r

def nb_params(unit):
    r = G.api_params(nb_payload({"k": unit}))
    if "error" in r: print("ERROR:", r["error"]); return r
    print(f"G{unit+1} ({r['tag']}) parameters:")
    for p in r["params"]:
        print(f"   {p['name']:10s} = {p['value']!s:>12}   {p['desc']}")
    return r

print("engine helpers ready — NB state:", NB)


## 3 · Native widget panel *(optional — needs `ipywidgets`)*

In [ ]:
# ---- native widget panel (optional; needs ipywidgets) ----
try:
    import ipywidgets as W
    from IPython.display import display
    _HAS_W = True
except ImportError:
    _HAS_W = False
    print('ipywidgets is not installed, so this optional panel is skipped.')
    print('To enable it:   pip install ipywidgets     (or: conda install ipywidgets)')
    print('The embedded lab (cell 1) and the nb_* helpers work without it.')
if _HAS_W:
    META = G.api_meta(None)
    _sys = W.Dropdown(options=[(v, k) for k, v in META["systems"].items()],
                      value=NB["system"], description="system")
    _mixbox = W.VBox([])
    _status = W.HTML()

    def _mk_mix():
        rows = []
        for k, tag in enumerate(NB["mix"]):
            dd = W.Dropdown(options=META["unit_types"], value=tag,
                            description=f"G{k+1}", layout=W.Layout(width="240px"))
            def _on(ch, k=k):
                if ch["name"] == "value":
                    NB["mix"][k] = ch["new"]; NB["prm"].pop(str(k), None); _refresh()
            dd.observe(_on)
            rows.append(dd)
        _mixbox.children = rows

    def _refresh():
        _status.value = (f"<b>{NB['system']}</b> [{', '.join(NB['mix'])}] · "
                         f"overrides on units: {sorted(NB['prm'].keys()) or 'none'}")

    def _on_sys(ch):
        if ch["name"] == "value":
            nb_set_system(ch["new"]); _mk_mix(); _refresh(); _mk_ov()
    _sys.observe(_on_sys)

    # parameter overrides
    _ov_unit = W.Dropdown(description="unit")
    _ov_par = W.Dropdown(description="param")
    _ov_val = W.FloatText(description="value")
    _ov_set = W.Button(description="set", button_style="primary",
                       layout=W.Layout(width="70px"))
    _ov_clr = W.Button(description="clear all", layout=W.Layout(width="90px"))

    def _mk_ov(*_):
        m = len(NB["mix"])
        _ov_unit.options = [(f"G{k+1} ({NB['mix'][k]})", k) for k in range(m)]

    def _ov_fill(ch=None):
        if _ov_unit.value is None: return
        r = G.api_params(nb_payload({"k": _ov_unit.value}))
        if "error" in r: return
        _ov_par.options = [(f"{p['name']} - {p['desc'][:44]}", p["name"])
                           for p in r["params"] if p["kind"] == "num"]
    _ov_unit.observe(lambda ch: ch["name"] == "value" and _ov_fill(), "value")

    def _ov_go(_):
        nb_set_override(_ov_unit.value, _ov_par.value, _ov_val.value); _refresh()
    _ov_set.on_click(_ov_go)
    _ov_clr.on_click(lambda _: (nb_clear_overrides(), _refresh()))

    # analysis tabs
    _out_ss = W.Output(); _out_td = W.Output(); _out_sw = W.Output()
    _b_lin = W.Button(description="Linearize & analyse", button_style="primary")
    _mode_k = W.BoundedIntText(value=0, min=0, max=40, description="mode #")
    _LAST = {}

    def _go_lin(_):
        with _out_ss:
            _out_ss.clear_output()
            _LAST["R"] = nb_linearize()
            if _LAST.get("R") and "modes" in _LAST["R"]:
                nb_mode_detail(_LAST["R"], min(_mode_k.value, len(_LAST["R"]["modes"]) - 1))
    _b_lin.on_click(_go_lin)
    def _go_mode(ch):
        if ch["name"] == "value" and _LAST.get("R"):
            with _out_ss:
                _out_ss.clear_output()
                nb_linearize()
                nb_mode_detail(_LAST["R"], min(ch["new"], len(_LAST["R"]["modes"]) - 1))
    _mode_k.observe(_go_mode)

    _td_kind = W.Dropdown(options=[("network: load change", "load"),
        ("network: 3-ph fault", "fault"), ("generator: set-point pulse", "gen"),
        ("source: cloud (PV unit)", "cloud"), ("source: gust (wind unit)", "gust")],
        description="class")
    _td_loc = W.IntText(value=8, description="bus/unit")
    _td_mag = W.FloatText(value=0.15, description="size")
    _td_t1 = W.FloatText(value=1.0, description="applied (s)")
    _td_t2 = W.Text(value="", description="removed (s)", placeholder="blank = sustained")
    _td_ts = W.FloatText(value=12.0, description="tsim (s)")
    _td_watch = W.Text(value="", description="watch", placeholder="Vdc3, SOC2, ...")
    _b_td = W.Button(description="Run simulation", button_style="primary")

    def _go_td(_):
        with _out_td:
            _out_td.clear_output()
            watch = [s.strip() for s in _td_watch.value.split(",") if s.strip()]
            nb_simulate(_td_kind.value, _td_loc.value, _td_mag.value,
                        _td_t1.value, _td_t2.value, _td_ts.value, watch)
    _b_td.on_click(_go_td)

    _sw_unit = W.Dropdown(description="unit")
    _sw_par = W.Dropdown(description="param")
    _sw_a = W.FloatText(value=2, description="from")
    _sw_b = W.FloatText(value=12, description="to")
    _sw_n = W.IntText(value=7, description="steps")
    _b_sw = W.Button(description="Sweep", button_style="primary")

    def _sw_units(*_):
        _sw_unit.options = [(f"G{k+1} ({NB['mix'][k]})", k) for k in range(len(NB["mix"]))]
    def _sw_fill(ch=None):
        if _sw_unit.value is None: return
        r = G.api_params(nb_payload({"k": _sw_unit.value}))
        if "error" in r: return
        _sw_par.options = [p["name"] for p in r["params"] if p["kind"] == "num"]
    _sw_unit.observe(lambda ch: ch["name"] == "value" and _sw_fill(), "value")

    def _go_sw(_):
        with _out_sw:
            _out_sw.clear_output()
            nb_sweep(_sw_unit.value, _sw_par.value, _sw_a.value, _sw_b.value, _sw_n.value)
    _b_sw.on_click(_go_sw)

    _tabs = W.Tab(children=[
        W.VBox([W.HBox([_b_lin, _mode_k]), _out_ss]),
        W.VBox([W.HBox([_td_kind, _td_loc, _td_mag]),
                W.HBox([_td_t1, _td_t2, _td_ts]),
                W.HBox([_td_watch, _b_td]), _out_td]),
        W.VBox([W.HBox([_sw_unit, _sw_par]),
                W.HBox([_sw_a, _sw_b, _sw_n, _b_sw]), _out_sw])])
    for i, t in enumerate(["Small-signal", "Time domain", "Parameter sweep"]):
        _tabs.set_title(i, t)

    _mk_mix(); _refresh(); _mk_ov(); _ov_fill(); _sw_units(); _sw_fill()
    _sys.observe(lambda ch: ch["name"] == "value" and (_mk_ov(), _sw_units()), "value")
    display(W.VBox([
        W.HTML("<h3 style='font-family:Georgia'>PSDAT — native notebook panel</h3>"),
        _sys, W.HTML("<b>unit mix</b> (one per machine):"), _mixbox,
        W.HTML("<b>parameter override</b> (any unit, any constant — no coding):"),
        W.HBox([_ov_unit, _ov_par, _ov_val, _ov_set, _ov_clr]),
        _status, _tabs]))


## 4 · Scripted example
Everything the GUI does is scriptable — ideal for assignments that must be reproducible:

In [ ]:
# ---- scripted use (works everywhere, no widgets needed) ----
# the same helpers drive scripted studies; a 60-second classroom example:
nb_set_system("IEEE9")
NB["mix"] = ["SG", "BESS-GFM", "PV-GFL"]           # re-equip the grid
nb_set_override(1, "Hv", 8.0)                        # a heavier virtual rotor
R = nb_linearize()                                   # exact modes + map
nb_mode_detail(R, 0)                                 # least-damped mode: who drives it?
nb_simulate("cloud", loc=3, mag=0.6, tsim=15,
            watch=["Vdc3", "SOC2"])                  # a cloud crosses the PV plant


---
**Notes.** The embedded lab and the widget panel share nothing but the engine, so results always
agree. On a remote JupyterHub the iframe may be blocked; open the printed `localhost` URL through
your port forwarding instead. State names for `watch` follow the manual (`Vdc·xdc·vref` PV,
`SOC·Pf` battery, `wt·wg·ttw·beta·Po` wind, `slip` induction, `delta·omega` SG — suffixed by bus
number). PSDAT — successor of PSDAT [Abdulrahman, IEEE OAJPE 2020].